# Experiment 46: Elite External Rank Blend

This experiment tests rank-based blends between Submission 14 and several high-scoring external predictions. The goal is to see whether small ranking corrections from strong external solutions can improve the current submission.

In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from scipy.stats import rankdata

print("=" * 100)
print("EXP46: ELITE EXTERNAL RANK BLEND")
print("=" * 100)

# This notebook lives in DataCompetition/notebooks/
# so the project root is one directory above the notebook folder.
ROOT = Path.cwd().parent

# If VS Code happens to launch from the project root instead,
# use it directly.
if not (ROOT / "external_data").exists():
    ROOT = Path.cwd()

S14 = ROOT / "submissions" / "submission_14.csv"

ZIP = (
    ROOT
    / "external_data"
    / "s6e9_zoom_zoom_baseline"
    / "ranked_predictions_latest.zip.bin"
)

CATALOG_PATH = (
    ROOT
    / "external_data"
    / "s6e9_zoom_zoom_baseline"
    / "ranked_catalog_latest.csv"
)

OUTPUT = ROOT / "submissions" / "experiment_46_blends"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Submission 14:", S14)
print("External archive:", ZIP)

if not S14.exists():
    raise FileNotFoundError(f"Submission 14 not found: {S14}")

if not ZIP.exists():
    raise FileNotFoundError(f"External archive not found: {ZIP}")

if not CATALOG_PATH.exists():
    raise FileNotFoundError(f"Catalog not found: {CATALOG_PATH}")

CATALOG = pd.read_csv(CATALOG_PATH)

s14 = pd.read_csv(S14)

print(f"Submission 14 rows: {len(s14):,}")

targets = {
    "amanatar": "public/amanatar__s6e9-grandmaster-sota-meta-blend/v350954426_fe53e31e248d.csv",
    "megayak": "public/megayak__s6e9-0-94656-reading-the-public-split/v350941977_23052891f418.csv",
    "nina": "public/nina2025__ps-s6e9-top-subm-will-buy-ev-round-4/v350957112_3d48f677f236.csv",
}

def load_prediction(zf, member):
    with zf.open(member) as f:
        return pd.read_csv(f)

with zipfile.ZipFile(ZIP) as zf:

    preds = {}

    for name, member in targets.items():
        preds[name] = load_prediction(zf, member)

print("Loaded external predictions:")

for name, df in preds.items():

    score = (
        CATALOG.loc[
            CATALOG["csv_path"] == targets[name],
            "public_score"
        ].iloc[0]
    )

    print(
        f"  {name:<9} "
        f"LB {score:.5f}"
    )

base_ids = s14["id"].to_numpy()

for name, df in preds.items():

    if not np.array_equal(
        df["id"].to_numpy(),
        base_ids
    ):
        raise RuntimeError(
            f"ID mismatch for {name}"
        )

base_rank = rankdata(
    s14["Will_Buy_EV"].to_numpy(dtype=np.float64)
)

rank_preds = {
    "s14": base_rank
}

for name, df in preds.items():

    rank_preds[name] = rankdata(
        df["Will_Buy_EV"].to_numpy(dtype=np.float64)
    )

def blend(weights):

    out = np.zeros(
        len(base_rank),
        dtype=np.float64
    )

    total = sum(weights.values())

    for name, weight in weights.items():
        out += rank_preds[name] * weight

    return out / total

recipes = {

    "blend_95_5_amanatar":
        {"s14": 95, "amanatar": 5},

    "blend_90_10_amanatar":
        {"s14": 90, "amanatar": 10},

    "blend_95_5_megayak":
        {"s14": 95, "megayak": 5},

    "blend_95_5_nina":
        {"s14": 95, "nina": 5},

    "blend_90_5_5_am_me":
        {"s14": 90, "amanatar": 5, "megayak": 5},

    "blend_90_5_5_am_ni":
        {"s14": 90, "amanatar": 5, "nina": 5},

    "blend_85_5_5_5":
        {"s14": 85, "amanatar": 5, "megayak": 5, "nina": 5},
}

print("\nCreating blends...")

summary = []

for name, weights in recipes.items():

    ranks = blend(weights)

    out = s14[["id"]].copy()

    out["Will_Buy_EV"] = (
        (ranks - ranks.min())
        / (ranks.max() - ranks.min())
    )

    path = OUTPUT / f"{name}.csv"

    out.to_csv(
        path,
        index=False
    )

    corr = np.corrcoef(
        rank_preds["s14"],
        ranks
    )[0, 1]

    mean_rank_shift = np.mean(
        np.abs(
            rank_preds["s14"] - ranks
        )
    )

    median_rank_shift = np.median(
        np.abs(
            rank_preds["s14"] - ranks
        )
    )

    summary.append({
        "blend": name,
        "corr_to_s14": corr,
        "mean_rank_shift": mean_rank_shift,
        "median_rank_shift": median_rank_shift,
        "file": path.name,
    })

summary = (
    pd.DataFrame(summary)
    .sort_values(
        "mean_rank_shift",
        ascending=False
    )
)

print("\n" + "=" * 100)
print("EXP46 BLEND SUMMARY")
print("=" * 100)

print(
    summary.to_string(index=False)
)

summary.to_csv(
    OUTPUT / "exp46_blend_summary.csv",
    index=False
)

print("\nSaved to:")
print(OUTPUT)

print("\n" + "=" * 100)
print("EXP46 COMPLETE")
print("=" * 100)